In [4]:
# importing necessary packages
import os
import time
import csv
import cv2
import mediapipe as mp
import pandas as pd

In [5]:
# reading training and validation JESTER metadata files
train_labels_df = pd.read_csv('/panfs/jay/groups/21/thyagara/sesha059/Hand-Gesture/train.csv')
val_labels_df = pd.read_csv('/panfs/jay/groups/21/thyagara/sesha059/Hand-Gesture/validation.csv')

In [6]:
# count of each gesture in training dataset
train_labels_df['gesture'].value_counts()

gesture
Doing other things               9592
Thumb Down                       4390
Thumb Up                         4373
Drumming Fingers                 4371
Pushing Hand Away                4357
Sliding Two Fingers Down         4348
Stop Sign                        4337
Zooming Out With Two Fingers     4323
Pulling Hand In                  4323
Zooming In With Two Fingers      4302
Sliding Two Fingers Left         4292
Pushing Two Fingers Away         4291
Zooming Out With Full Hand       4281
No gesture                       4278
Pulling Two Fingers In           4267
Swiping Down                     4259
Shaking Hand                     4258
Zooming In With Full Hand        4251
Swiping Up                       4220
Sliding Two Fingers Up           4219
Sliding Two Fingers Right        4206
Swiping Left                     4162
Rolling Hand Forward             4132
Swiping Right                    4084
Rolling Hand Backward            4032
Turning Hand Counterclockwise    3398
Turn

In [7]:
# count of each gesture in validation dataset
val_labels_df['gesture'].value_counts()

gesture
Doing other things               1468
Thumb Up                          539
Pushing Hand Away                 538
Stop Sign                         536
Thumb Down                        536
Drumming Fingers                  535
No gesture                        533
Sliding Two Fingers Down          531
Pushing Two Fingers Away          531
Zooming Out With Two Fingers      530
Shaking Hand                      528
Zooming In With Full Hand         526
Pulling Hand In                   526
Zooming Out With Full Hand        526
Sliding Two Fingers Up            522
Zooming In With Two Fingers       522
Rolling Hand Forward              521
Swiping Down                      520
Sliding Two Fingers Left          519
Pulling Two Fingers In            519
Sliding Two Fingers Right         516
Swiping Up                        508
Swiping Left                      494
Rolling Hand Backward             493
Swiping Right                     486
Turning Hand Counterclockwise     399
Turn

In [8]:
# function to extract landmarks from a give image
def extract_hand_landmarks(image, video_name, gesture_label, frame_number):
    """Extract hand landmarks from an image and normalize X, Y coordinates relative to the wrist."""
    results = hands.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    if not results.multi_hand_landmarks:
        return None

    landmarks_list = []
    for hand_landmarks in results.multi_hand_landmarks:
        wrist_x = hand_landmarks.landmark[mp_hands.HandLandmark.WRIST].x
        wrist_y = hand_landmarks.landmark[mp_hands.HandLandmark.WRIST].y

        landmarks = [(landmark.x - wrist_x, landmark.y - wrist_y, landmark.z)
                     for landmark in hand_landmarks.landmark]
        flattened_landmarks = [coordinate for landmark in landmarks for coordinate in landmark]
        landmarks_list.extend(flattened_landmarks)

    return [video_name, gesture_label, frame_number] + landmarks_list


# function to save the extracted landmark information to a csv file
def save_features_to_csv(features, csv_path):
  with open(csv_path, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    #writer.writerow(['video_name', 'gesture_label', 'frame_number'] + [f'landmark_{i}' for i in range(21 * 3)])
    for feature in features:
      writer.writerow(feature)
    
    
# helper-function to processes a video with a given gesture label in video directory
def process_video(video_directory, gesture_label):
  frame_number = 0
  features = []
  video_name = video_directory.split('/')[-2]

  # Loop through JPG images in the directory
  for filename in os.listdir(video_directory):
    if not filename.endswith('.jpg'):
      continue

    # Extract frame number from filename
    frame_number = int(filename.split('.')[0])

    # Load image
    image = cv2.imread(os.path.join(video_directory, filename))

    # Extract hand landmarks
    landmarks = extract_hand_landmarks(image, video_name, gesture_label, frame_number)
    if landmarks is None:
      continue
    
    features.append(landmarks)

  return features

In [9]:
# sorting by video_num in train_lables_df
train_labels_df = train_labels_df.sort_values(by=['video_num'])
train_labels_df.reset_index(inplace=True, drop=True)

# getting video_num for already extracted ones
csv_path = '/panfs/jay/groups/21/thyagara/sesha059/Hand-Gesture/train_feature_maps.csv'
feature_maps_df = pd.read_csv(csv_path)
previous_runs = list(feature_maps_df['video_name'].unique())

# getting yet to be extracted ones
yet_to_be_df = train_labels_df.loc[~train_labels_df['video_num'].isin(previous_runs), :]

# replace train_labels_df with val_labels_df and train_feature_maps with val_feature_maps to extract landmarks for validation dataset

In [10]:
# printing the count of videos for which landmarks were already extracted
len(previous_runs)

53128

In [ ]:
# initializing mediapipe hands recognition function
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.4, min_tracking_confidence=0.4, max_num_hands=1)

# start time of the landmark extraction task
start_time = time.time()
for index, row in yet_to_be_df.iterrows():
    video_num = row['video_num']
    gesture_label = row['gesture']
    video_directory = "/scratch.global/Number20/JESTER/20bn-jester-v1/"+str(video_num)+"/"
    
    # checking if landmarks for the specific video is already extracted
    if video_num not in previous_runs:
        try:
            # processes a video to landmarks for all of its frames
            features = process_video(video_directory = video_directory, gesture_label = gesture_label)
            # appends the extracted landmarks list to the given csv file
            save_features_to_csv(features=features, csv_path=csv_path)            
        except:
            pass
    
    if index%500 == 0:
        # end time when landmarks for 500 videos got extracted
        end_time = time.time()
        time_diff = end_time-start_time
        print(f"Number of videos processed: {index+1}, time taken: {time_diff} seconds")

I0000 00:00:1702930954.084104  583573 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1702930954.186547  583995 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 470.103.01), renderer: NVIDIA A40/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Number of videos processed: 501, time taken: 48.8989634513855 seconds
Number of videos processed: 1501, time taken: 139.72120475769043 seconds
Number of videos processed: 2501, time taken: 225.43140959739685 seconds
Number of videos processed: 4501, time taken: 395.0459840297699 seconds
Number of videos processed: 6501, time taken: 546.1086905002594 seconds
Number of videos processed: 8001, time taken: 670.2760488986969 seconds
Number of videos processed: 8501, time taken: 703.9286966323853 seconds
Number of videos processed: 9501, time taken: 778.1471664905548 seconds
Number of videos processed: 12501, time taken: 1000.5502490997314 seconds
Number of videos processed: 13001, time taken: 1057.1328880786896 seconds
Number of videos processed: 13501, time taken: 1088.9916095733643 seconds
Number of videos processed: 14501, time taken: 1182.7517731189728 seconds
Number of videos processed: 15001, time taken: 1216.5336530208588 seconds
Number of videos processed: 15501, time taken: 1250.17